# 02 · LLM-as-judge and its biases

After this notebook you can build three kinds of LLM judge with structured output: rubric, pairwise and reference-guided. You can measure a judge's position, verbosity and self-preference bias with controlled tests, check it against human labels with Cohen's kappa (coded from scratch), and estimate what it costs to run.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · eval metrics](01_llm_eval_metrics_em_f1_bleu_rouge.ipynb), [Structured outputs with Pydantic](../08_llm_apps/03_structured_outputs_with_pydantic.ipynb)

## Why this matters in interviews

- LLM-as-judge is the default way to grade open-ended output. The standard follow-up is "why would you trust it?" (`ev02`), and the answer has to include a number: agreement with humans.
- "Name the judge biases and the fixes" (`ev03`) is a near-certain question. Strong candidates also say how they would **measure** each bias, not just name it.
- Pairwise versus absolute scoring, Cohen's kappa and judge drift are the usual depth probes.
- Swap tests and self-consistency multiply the judge bill. Being able to estimate that cost on a whiteboard is a senior signal.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ev02` | If LLMs hallucinate, why would you trust another LLM to evaluate them? |
| `ev03` | How do you make an LLM judge reliable? Name the biases and the fixes. |
| `ev04` | Compare the main evaluation methods: exact match, semantic similarity, LLM judge, pairwise, RAGAS, human, and online A/B. |
| `ev08` | What is pairwise evaluation and why is it more reliable than scoring? |
| `ev18` | How do you evaluate when there is no single right answer? |
| `ev19` | What is Cohen's kappa and why does it matter here? |
| `ev34` | How do you know your judge has not drifted? |
| `ev16` | How do you evaluate a summarisation feature? |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
from dataclasses import dataclass
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import openai
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from sklearn.metrics import cohen_kappa_score, confusion_matrix

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 60)
rng = np.random.default_rng(0)

## Three ways to ask a judge

| Format | You send | You get | Best for | Weakness |
|---|---|---|---|---|
| **Rubric (pointwise)** | question + answer + a rubric with a written definition per score | a score 1–5 and a reason | monitoring one system over time, thresholds in CI | scores bunch at 4–5; the scale drifts |
| **Pairwise** | question + answer A + answer B | A, B or tie | "is version 2 better than version 1?" (`ev08`) | position bias; gives a ranking, not a level |
| **Reference-guided** | question + reference answer + candidate | correct / partial / incorrect | factual tasks with a known answer | needs references; judge may over-trust the reference wording |

**In plain English:** judging is easier than writing, just as spotting the better of two essays is easier than writing one (`ev02`). You can also give the judge advantages the generator lacked: a rubric, a reference, the source documents. What makes it *trustworthy* is not the model, though. It is that you **measure** its agreement with humans, which is the second half of this notebook.

## 1. Rubric grading with structured output

Put the rubric in the system prompt and ask for a Pydantic object. Put `reasoning` **before** `score`: with OpenAI structured outputs, keys are generated in schema order, so the model writes its justification before it commits to a number. The helper catches parse failures instead of crashing. A judge that returns an out-of-range score is a real production event, and you should count it.

In [ ]:
class Grade(BaseModel):
    reasoning: str = Field(description="One or two sentences that cite the rubric.")
    score: int = Field(ge=1, le=5, description="1-5 as defined by the rubric.")

RUBRIC = """You grade answers from a customer-support assistant. Use this rubric:
5 - correct and complete; nothing important missing
4 - correct, but misses a minor detail
3 - partly correct, or correct but padded with irrelevant text
2 - mostly incorrect, or evades the question
1 - wrong, unsafe, or off-topic
Length is not quality: never reward an answer for being longer. Write your reasoning first, then the score."""

def judge_call(messages, schema):
    """One structured-output judge call -> (parsed object or None, error name or None)."""
    try:
        resp = client.chat.completions.parse(model=MODEL, messages=messages, response_format=schema, temperature=0)
        msg = resp.choices[0].message
        return (msg.parsed, None) if msg.parsed is not None else (None, msg.refusal or "empty")
    except (ValidationError, openai.LengthFinishReasonError, openai.ContentFilterFinishReasonError) as e:
        return None, type(e).__name__

def judge_rubric(question, answer):
    return judge_call([{"role": "system", "content": RUBRIC},
                       {"role": "user", "content": f"QUESTION: {question}\nANSWER TO GRADE: {answer}"}], Grade)

rubric_cases = [("What is the capital of Australia?", "Canberra."),
                ("What is the capital of Australia?", "Sydney, the biggest and best-known city."),
                ("How do I reset my password?", "Great question! Passwords matter. Security is important to us.")]
for q, a in rubric_cases:
    grade, err = judge_rubric(q, a)
    print(f"{a[:45]:47} -> " + (f"score={grade.score}  reason={grade.reasoning[:70]!r}" if grade else f"PARSE FAILURE ({err})"))
if PROVIDER == "offline":
    print("\n(offline stand-in: it fills the schema from the prompt and does not read the rubric, so these scores are "
          "canned. A real judge should give roughly 5, 1 and 2.)")

## 2. Pairwise comparison, and the swap test

Ask "which is better, A or B?", then ask again with the two answers **swapped**. A trustworthy verdict follows the *answer*, not the *slot*. If it says "A" both times, it preferred whatever came first. That is position bias, and the swap test needs **no human labels** to detect it. The standard fix (`ev03`) is to run both orders and keep only verdicts that agree, counting a flip as a tie. That costs exactly 2× the calls.

In [ ]:
class PairVerdict(BaseModel):
    reasoning: str = Field(description="Compare the two answers against the question in one or two sentences.")
    winner: Literal["A", "B", "tie"]

PAIR_SYSTEM = ("You compare two answers to the same customer question. Prefer the answer that is correct and "
               "addresses the question. Ignore length and style. Say 'tie' if they are equally good.")

def judge_pair(question, first, second):
    user = f"QUESTION: {question}\n\n[ANSWER A]\n{first}\n\n[ANSWER B]\n{second}\n\nWhich answer is better: A, B or tie?"
    return judge_call([{"role": "system", "content": PAIR_SYSTEM}, {"role": "user", "content": user}], PairVerdict)

PAIRS = [  # (question, x, y, what a careful human says)
    ("How do I reset my password?", "Click 'Forgot password' on the login page and follow the link we email you.", "Contact support.", "x"),
    ("What is the capital of Australia?", "Sydney.", "Canberra.", "y"),
    ("Can I get a refund after sixty days?", "No. Refunds are only available within thirty days of delivery.", "Yes, refunds are always available.", "x"),
    ("What is the capital of Australia?", "Canberra.", "Great question! Australia has famous cities such as Sydney and Melbourne. "
     "After careful consideration, the capital is Canberra.", "tie (same fact; y is padded)"),
]

def resolve(v_xy, v_yx):
    """Map the two verdicts back to answers: x/y if both orders agree, otherwise 'tie (flipped)'."""
    first = {"A": "x", "B": "y", "tie": "tie"}.get(v_xy)        # order 1 shows x first
    second = {"A": "y", "B": "x", "tie": "tie"}.get(v_yx)       # order 2 shows y first
    return first if first == second else "tie (flipped)"

rows = []
for q, x, y, human in PAIRS:
    v1, _ = judge_pair(q, x, y)
    v2, _ = judge_pair(q, y, x)
    w1, w2 = (v1.winner if v1 else "fail"), (v2.winner if v2 else "fail")
    rows.append({"question": q, "x shown first": w1, "y shown first": w2, "consistent": resolve(w1, w2) != "tie (flipped)",
                 "resolved": resolve(w1, w2), "human": human})
swap = pd.DataFrame(rows)
print(swap.to_string(index=False))
print(f"\nposition-consistent verdicts: {swap.consistent.sum()} of {len(swap)}")
if PROVIDER == "offline":
    print("(offline stand-in always answers 'A': the most position-biased judge possible. The swap test flags every pair,\n"
          " which is exactly what it is for. A real gpt-4.1-mini judge is usually consistent on clear-cut pairs like these.)")

## 3. Reference-guided grading

When a correct answer exists, give it to the judge. The judge then only has to decide whether the candidate *says the same thing*. That is far easier than knowing the fact, and it catches the paraphrases that token F1 and BLEU missed in [notebook 01](01_llm_eval_metrics_em_f1_bleu_rouge.ipynb). Use three labels, not a 1–10 scale: categorical verdicts are more stable (`ev18`).

In [ ]:
class RefVerdict(BaseModel):
    reasoning: str = Field(description="Which facts of the reference are present, missing or contradicted.")
    verdict: Literal["correct", "partially_correct", "incorrect"]

REF_SYSTEM = ("Decide whether the CANDIDATE answer agrees with the REFERENCE answer. Wording does not matter; facts do. "
              "Contradicting any fact in the reference makes it incorrect.")
REF_CASES = [("What is the capital of Australia?", "Canberra is the capital of Australia.", "It's Canberra."),
             ("What is the capital of Australia?", "Canberra is the capital of Australia.", "Sydney, the largest city."),
             ("Can I get a refund after sixty days?", "No: refunds are only available within thirty days of delivery.",
              "Only within a month of delivery, so no.")]
for q, ref, cand in REF_CASES:
    v, err = judge_call([{"role": "system", "content": REF_SYSTEM},
                         {"role": "user", "content": f"QUESTION: {q}\nREFERENCE: {ref}\nCANDIDATE: {cand}"}], RefVerdict)
    print(f"{cand:42} -> {v.verdict if v else 'PARSE FAILURE ' + str(err)}")
if PROVIDER == "offline":
    print("\n(offline: canned. A real judge should say correct, incorrect, correct. The third is the paraphrase "
          "that lexical metrics score near zero.)")

## 4. The biases, and how to measure each one

| Bias | What it looks like | How to **measure** it | Fix |
|---|---|---|---|
| **Position** | the first (or second) answer wins too often | swap test: first-slot win rate, % of pairs whose verdict survives swapping | run both orders, keep agreeing verdicts, count flips as ties |
| **Verbosity** | longer answers win | length-controlled test: same content plus padding, both orders | tell it length is not quality; track verdict-vs-length; length-matched pairs |
| **Self-preference** | a model favours its own family's outputs | own-family win rate by the judge vs by humans on the same pairs | judge with a different model family |
| **Leniency / poor calibration** | scores bunch at 4–5 | score histogram and confusion matrix against human labels | anchored rubric with a definition per score; prefer pairwise |
| **Sampling noise** | same input, different score | re-run the same items and measure the spread | temperature 0, or majority of n = 3 |
| **Drift** (`ev34`) | the same answers score differently next month | re-run a frozen, human-labelled calibration set | pin the judge model version **and** the judge prompt |

The offline stand-in is not a judge, so we cannot measure *its* biases meaningfully. Instead we build a **transparent simulated judge** whose biases we set ourselves, with one dial per bias. Then we check that each measurement recovers the bias we put in. That is how you validate a methodology before you trust it on a real judge. With a real API key, the same test functions run unchanged on a real judge, just more slowly and for money.

In [ ]:
@dataclass
class SimulatedJudge:
    """A stand-in LLM judge with one dial per known bias (0 = no bias)."""
    skill: float = 1.6            # how strongly true quality drives verdicts
    position_bias: float = 0.0    # extra logit for whichever answer is shown FIRST
    verbosity_bias: float = 0.0   # extra logit (pairwise) or points (pointwise) per 100 tokens of length
    self_preference: float = 0.0  # extra logit / points for answers written by the judge's own model family
    leniency: float = 0.0         # pointwise only: points added to every score
    noise: float = 0.6            # pointwise only: sd of score noise
    seed: int = 0

    def __post_init__(self):
        self.rng = np.random.default_rng(self.seed)

    def logit_first(self, first, second):
        return (self.skill * (first.quality.values - second.quality.values)
                + self.verbosity_bias * (first.length.values - second.length.values) / 100
                + self.self_preference * (first.own.values.astype(int) - second.own.values.astype(int))
                + self.position_bias)

    def compare(self, first: pd.DataFrame, second: pd.DataFrame) -> np.ndarray:
        """Pairwise verdicts for aligned rows: True where the answer shown FIRST wins."""
        return self.rng.random(len(first)) < 1 / (1 + np.exp(-self.logit_first(first, second)))

    def score(self, ans: pd.DataFrame) -> np.ndarray:
        """Pointwise 1-5 scores."""
        raw = (ans.quality.values + self.leniency + self.verbosity_bias * (ans.length.values - 150) / 100
               + self.self_preference * ans.own.values + self.rng.normal(0, self.noise, len(ans)))
        return np.clip(np.round(raw), 1, 5).astype(int)

The simulated world: 400 answers, each with a true quality (1–5), a length in tokens, and a flag for whether the judge's own model family wrote it. Better answers run a little longer, as in real data. That is why a naive "longer answers win" correlation cannot separate a verbosity bias from genuine quality. Two human annotators label every answer. Their labels are the true quality plus rounding noise.

In [ ]:
N = 400
quality = rng.integers(1, 6, N)
answers = pd.DataFrame({
    "quality": quality,
    "length": np.clip(40 + 30 * quality + rng.normal(0, 45, N), 15, None).round(),
    "own": rng.random(N) < 0.5,
    "human1": np.clip(np.round(quality + rng.normal(0, 0.5, N)), 1, 5).astype(int),
    "human2": np.clip(np.round(quality + rng.normal(0, 0.5, N)), 1, 5).astype(int),
})
print(answers.head().to_string())
print(f"\ncorrelation(length, true quality) = {np.corrcoef(answers.length, answers.quality)[0, 1]:.2f}")

i = rng.integers(0, N, 1500); j = rng.integers(0, N, 1500)
keep = i != j
A, B = answers.iloc[i[keep]].reset_index(drop=True), answers.iloc[j[keep]].reset_index(drop=True)
print(f"{len(A)} random answer pairs for the pairwise tests")

### 4a. Position bias: the swap test

Judge every pair in both orders and compute two numbers. **First-slot win rate** should be 0.5 for an unbiased judge. **Consistency** is the share of pairs where the same answer wins in both orders. The measurement uses no labels at all.

In [ ]:
def swap_test(judge, A, B):
    ab = judge.compare(A, B)            # True: A (shown first) wins
    ba = judge.compare(B, A)            # True: B (shown first) wins
    return {"first_win_rate": (ab.mean() + ba.mean()) / 2, "consistency": (ab != ba).mean(), "ab": ab, "ba": ba}

biases = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0]
results = []
for pb in biases:
    judge = SimulatedJudge(position_bias=pb, seed=1)
    r = swap_test(judge, A, B)
    sig = lambda z: 1 / (1 + np.exp(-z))
    expected = (sig(judge.logit_first(A, B)).mean() + sig(judge.logit_first(B, A)).mean()) / 2
    results.append({"position_bias": pb, "first_win_rate": r["first_win_rate"], "expected": expected, "consistency": r["consistency"]})
res = pd.DataFrame(results)
print(res.round(3).to_string(index=False))

assert abs(res.first_win_rate[0] - 0.5) < 0.03                      # unbiased judge: first slot wins half the time
assert (abs(res.first_win_rate - res.expected) < 0.03).all()         # the swap test recovers the bias we injected
assert res.consistency[0] > res.consistency[3] > res.consistency[5]  # more bias -> more flipped verdicts

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(res.position_bias, res.first_win_rate, "o-", label="first-slot win rate (0.5 = fair)")
ax.plot(res.position_bias, res.consistency, "s-", label="consistency after swapping")
ax.axhline(0.5, ls="--", color="gray", lw=1)
ax.set_xlabel("injected position bias (extra logit for the first answer)"); ax.set_ylabel("rate")
ax.set_title("Swap test: no labels needed to see position bias"); ax.legend(); plt.show()

**Does "keep only consistent verdicts" help?** Score each verdict against the truth (the pair member with higher true quality), on pairs whose quality differs. Compare a single-order verdict with the swap-and-keep-agreeing rule. The price is **coverage**: flipped pairs become ties, and you have paid for two calls per pair.

In [ ]:
judge = SimulatedJudge(position_bias=1.0, seed=2)
r = swap_test(judge, A, B)
differs = A.quality.values != B.quality.values
a_better = A.quality.values > B.quality.values
single = (r["ab"] == a_better)[differs].mean()
kept = (r["ab"] != r["ba"]) & differs
swapped = (r["ab"] == a_better)[kept].mean()
print(f"single order     : accuracy {single:.3f} on 100% of pairs")
print(f"swap, keep agree : accuracy {swapped:.3f} on {kept.sum() / differs.sum():.0%} of pairs (the rest become ties)")
assert swapped > single

### 4b. Verbosity bias: the length-controlled (padding) test

Correlating verdicts with length is confounded here, because better answers are longer. So hold the content fixed instead: compare each answer with **itself plus padding**. True quality is identical, so a fair judge picks the padded copy half the time. Run both orders so any position bias cancels out.

In [ ]:
base = pd.concat([answers] * 3, ignore_index=True)        # every answer, 3 repeats -> 1,200 comparisons per order
pads = [0, 25, 50, 100, 200]
fig, ax = plt.subplots(figsize=(8, 3.8))
padding_results = {}
for vb in [0.0, 0.4, 0.8]:
    judge = SimulatedJudge(verbosity_bias=vb, position_bias=0.8, seed=3)   # position bias present on purpose
    rates = []
    for pad in pads:
        padded = base.assign(length=base.length + pad)
        rates.append(float((judge.compare(padded, base).mean() + (~judge.compare(base, padded)).mean()) / 2))
    padding_results[vb] = rates
    ax.plot(pads, rates, "o-", label=f"injected verbosity bias {vb}")
ax.axhline(0.5, ls="--", color="gray", lw=1)
ax.set_xlabel("padding added (tokens of fluff, same content)"); ax.set_ylabel("P(padded copy wins)")
ax.set_title("Length-controlled test: a fair judge stays at 0.5"); ax.legend(); plt.show()
print({vb: [round(x, 2) for x in rates] for vb, rates in padding_results.items()})

assert all(abs(x - 0.5) < 0.06 for x in padding_results[0.0])     # no verbosity bias -> flat at 0.5
assert padding_results[0.8][-1] > 0.75 and padding_results[0.8][-1] > padding_results[0.4][-1]

### 4c. Self-preference

Take pairs with one answer from the judge's own model family and one from another family, and keep only the pairs that **humans rated equal**. On those pairs a fair judge should pick its own family half the time, so any excess is self-preference. Matching on the human label is the control, just as padding the *same* answer was the control for verbosity. Without it, a chance quality difference between the two families would look like bias.

In [ ]:
own_i = rng.choice(answers.index[answers.own], 3000); other_i = rng.choice(answers.index[~answers.own], 3000)
O, X = answers.loc[own_i].reset_index(drop=True), answers.loc[other_i].reset_index(drop=True)
tied = O.human1.values == X.human1.values                          # humans say: equally good
O, X = O[tied].reset_index(drop=True), X[tied].reset_index(drop=True)
print(f"{len(O)} own-vs-other pairs that humans rated equal")
for sp in [0.0, 0.5]:
    judge = SimulatedJudge(self_preference=sp, position_bias=0.8, seed=4)
    judge_own = (judge.compare(O, X).mean() + (~judge.compare(X, O)).mean()) / 2     # both orders
    print(f"injected self_preference={sp}: judge picks its own family {judge_own:.3f} of the time (fair = 0.500)")
    if sp == 0.0:
        assert abs(judge_own - 0.5) < 0.04
    else:
        assert judge_own - 0.5 > 0.06

### 4d. Leniency, and agreement with humans: Cohen's kappa

Raw agreement flatters: if two raters both say "good" 90% of the time, they agree at least 82% of the time by pure chance. **Cohen's kappa** removes that chance agreement (`ev19`):

κ = (p_o − p_e) / (1 − p_e), where p_o is the observed agreement and p_e = Σ_k (row share_k × column share_k) is the agreement expected by chance.

For ordinal scores like 1–5, the **weighted** kappa gives partial credit for near misses: 4 vs 5 is not as bad as 1 vs 5. Rough reading: < 0.4 poor, 0.4–0.6 moderate, 0.6–0.8 substantial, > 0.8 excellent. Here is the worked example by hand: 100 items, 85 both good, 5 + 5 disagreements, 5 both bad. So p_o = 0.90, p_e = 0.9·0.9 + 0.1·0.1 = 0.82, and κ = 0.08 / 0.18 = **0.444**. That is 90% raw agreement but only "moderate" kappa.

In [ ]:
def cohen_kappa(a, b, weights=None):
    """Cohen's kappa from scratch; weights=None, 'linear' or 'quadratic' (ordinal labels)."""
    labels = sorted(set(a) | set(b))
    idx = {l: k for k, l in enumerate(labels)}
    K = len(labels)
    O = np.zeros((K, K))
    for x, y in zip(a, b):
        O[idx[x], idx[y]] += 1
    O /= O.sum()
    E = np.outer(O.sum(axis=1), O.sum(axis=0))              # chance agreement from the two raters' marginals
    d = np.abs(np.subtract.outer(np.arange(K), np.arange(K)))
    W = {None: (d > 0).astype(float), "linear": d, "quadratic": d ** 2}[weights]   # disagreement weights
    return 1 - (W * O).sum() / (W * E).sum()

a = ["good"] * 85 + ["good"] * 5 + ["bad"] * 5 + ["bad"] * 5
b = ["good"] * 85 + ["bad"] * 5 + ["good"] * 5 + ["bad"] * 5
print(f"raw agreement {np.mean(np.array(a) == np.array(b)):.2f}, kappa {cohen_kappa(a, b):.3f}")
assert abs(cohen_kappa(a, b) - 0.08 / 0.18) < 1e-12 and abs(cohen_kappa(a, b) - cohen_kappa_score(a, b)) < 1e-12

In [ ]:
raw_judge = SimulatedJudge(leniency=0.8, verbosity_bias=0.6, self_preference=0.5, noise=0.6, seed=5)
careful_judge = SimulatedJudge(leniency=0.0, verbosity_bias=0.1, self_preference=0.0, noise=0.6, seed=5)
scores = {"human 2": answers.human2.values, "raw judge": raw_judge.score(answers), "careful judge": careful_judge.score(answers)}
h1 = answers.human1.values
rows = []
for name, s in scores.items():
    row = {"vs human 1": name, "mean score": s.mean(), "raw agreement": (s == h1).mean()}
    for w in [None, "quadratic"]:
        k = cohen_kappa(h1, s, w)
        assert abs(k - cohen_kappa_score(h1, s, weights=w)) < 1e-12       # from-scratch == sklearn
        row["kappa" if w is None else "kappa (quadratic)"] = k
    rows.append(row)
agree = pd.DataFrame(rows).set_index("vs human 1")
print(f"human 1 mean score: {h1.mean():.2f}")
print(agree.round(3).to_string())
assert agree.loc["raw judge", "mean score"] > h1.mean() + 0.4                    # leniency is visible
assert agree.loc["careful judge", "kappa (quadratic)"] > agree.loc["raw judge", "kappa (quadratic)"]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
cm = confusion_matrix(h1, scores["raw judge"], labels=[1, 2, 3, 4, 5])
axes[0].imshow(cm, cmap="Blues")
for r_ in range(5):
    for c_ in range(5):
        axes[0].text(c_, r_, cm[r_, c_], ha="center", va="center", color="white" if cm[r_, c_] > cm.max() / 2 else "black")
axes[0].set_xticks(range(5), range(1, 6)); axes[0].set_yticks(range(5), range(1, 6))
axes[0].set_xlabel("raw judge score"); axes[0].set_ylabel("human score")
axes[0].set_title("Mass right of the diagonal = a lenient judge")
x = np.arange(1, 6)
for k, (name, s) in enumerate([("human 1", h1), ("raw judge", scores["raw judge"]), ("careful judge", scores["careful judge"])]):
    axes[1].bar(x + (k - 1) * 0.27, np.bincount(s, minlength=6)[1:], width=0.27, label=name)
axes[1].set_xlabel("score"); axes[1].set_ylabel("count"); axes[1].legend()
axes[1].set_title("The raw judge bunches at 4-5")
plt.tight_layout(); plt.show()

**Reading the numbers:**

- Human-vs-human kappa is the **reference point**. A judge that agrees with one human about as well as a second human does is as good as you can measure with these labels. If the humans themselves agree poorly, fix the labelling guideline before evaluating anything (`ev19` trap).
- The raw judge's unweighted kappa is low even though it "mostly gets the direction right". Leniency and length bias shift whole columns of the confusion matrix.
- After you change the judge (an anchored rubric, a different model family, "length is not quality"), you **re-run the same calibration set**. Here we emulate that change by turning the dials down. With a real judge, the before/after kappa is simply what you measure. Around 0.6 or above is the usual bar for trusting it as a regression detector (`ev02`).
- Keep that calibration set frozen and re-run it whenever the judge model, its version or its prompt changes (`ev34`).

## 5. Mitigations at a glance

| Problem | Mitigation | Cost |
|---|---|---|
| Position bias | both orders, keep agreeing verdicts, flips = ties | 2× calls |
| Verbosity bias | "length is not quality" in the prompt; length-matched pairs; monitor verdict-vs-length | free / small |
| Self-preference | judge from a different model family than the generator | none, but a second vendor |
| Leniency, scale drift | rubric with a written anchor per score; binary or 3-way labels; pairwise instead of 1–10 | prompt work |
| Sampling noise | temperature 0; majority vote of n = 3 | up to 3× calls |
| No idea if it works | 50–100 human-labelled cases, report kappa before trusting any number | one day of labelling |
| Drift | pin model **version** + prompt; re-run the calibration set on any change | small, recurring |
| Judging the wrong thing | decompose "good" into named yes/no criteria (`ev18`) | more calls, much more signal |

## 6. What does the judge cost?

Count the tokens of the actual prompts. The judge's price is input tokens (rubric + question + answers) plus output tokens (reasoning + verdict), times the number of calls your mitigations add. The prices below are gpt-4.1-mini list prices at the time of writing. Check the pricing page before quoting them.

In [ ]:
try:
    import tiktoken
    enc = tiktoken.get_encoding("o200k_base")          # the tokenizer of the gpt-4o / gpt-4.1 family
    count = lambda s: len(enc.encode(s))
except Exception:
    count = lambda s: max(1, len(s) // 4)               # rough fallback: ~4 characters per token

PRICE = {"input": 0.40, "output": 1.60}               # USD per 1M tokens (gpt-4.1-mini list price; check current)
q, x, y, _ = PAIRS[0]
pointwise_in = count(RUBRIC) + count(f"QUESTION: {q}\nANSWER TO GRADE: {x}") + 60      # +60 ~ a longer real answer
pairwise_in = count(PAIR_SYSTEM) + count(f"QUESTION: {q}\n[ANSWER A]\n{x}\n[ANSWER B]\n{y}") + 120
out_tokens = 60                                                                          # reasoning + verdict
strategies = {"pointwise": (pointwise_in, 1), "pairwise": (pairwise_in, 1),
              "pairwise + swap": (pairwise_in, 2), "pairwise + swap + n=3 votes": (pairwise_in, 6)}
cases, runs_per_month = 300, 30 + 40                                                     # nightly + ~40 PR runs
rows = []
for name, (tin, calls) in strategies.items():
    per_call = (tin * PRICE["input"] + out_tokens * PRICE["output"]) / 1e6
    rows.append({"strategy": name, "input tokens/call": tin, "calls/case": calls,
                 "$ per 1k cases": 1000 * calls * per_call, "$ per month": cases * runs_per_month * calls * per_call})
cost = pd.DataFrame(rows).set_index("strategy")
print(cost.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(cost.index, cost["$ per month"], color="#3b5bdb")
ax.set_xlabel(f"USD per month ({cases} cases x {runs_per_month} runs)")
ax.set_title("Every mitigation multiplies the bill; the absolute numbers stay small on a mini model"); plt.show()

Rules of thumb: a bigger judge model costs roughly 5× more per token than the mini tier. The swap test is worth its 2× almost always. Run the full judged suite nightly and a small subset per PR (`ev28`).

## Try it yourself

**1. How many pairs do you need?** A judge with `position_bias=0.3` is subtly biased. For 25, 50, 100, 200 and 400 pairs, run the swap test and compute a 95% confidence interval for the first-slot win rate: rate ± 1.96·√(rate(1−rate)/n_judgments). At what size does the interval exclude 0.5? (More on this in [notebook 05](05_ab_testing_and_statistics_for_evals.ipynb).)

In [ ]:
# Solution — try it yourself first, then run this
judge = SimulatedJudge(position_bias=0.3, seed=7)
for n in [50, 100, 200, 400, 800, len(A)]:
    rate = swap_test(judge, A.iloc[:n], B.iloc[:n])["first_win_rate"]
    half = 1.96 * np.sqrt(rate * (1 - rate) / (2 * n))
    print(f"{n:4d} pairs: first-slot win rate {rate:.3f} ± {half:.3f}  -> {'bias detected' if rate - half > 0.5 else 'cannot tell'}")
# A subtle bias needs on the order of a thousand swapped pairs to show up reliably; a 50-pair audit cannot see it.

**2. Self-consistency.** Score every answer 3 times with the noisy `raw_judge` and take the median. How much does quadratic kappa against human 1 improve? What does it cost?

In [ ]:
# Solution — try it yourself first, then run this
single = raw_judge.score(answers)
median3 = np.median(np.stack([raw_judge.score(answers) for _ in range(3)]), axis=0).astype(int)
print(f"1 sample : kappa (quadratic) {cohen_kappa(h1, single, 'quadratic'):.3f}")
print(f"median 3 : kappa (quadratic) {cohen_kappa(h1, median3, 'quadratic'):.3f}   (3x the calls)")
# It barely moves: this judge's error is mostly bias (leniency + length), and voting removes noise, not bias.
# The lenient offset survives the median. Fix bias with the rubric and the controls above.

**3. From pairwise verdicts to a ranking.** Four systems have true strengths [0, 0.5, 1.0, 1.5] (logits). Simulate 600 random pairwise judgments with random display order and a position bias of 0.8. Fit Elo ratings (K = 16) and check that the ranking is right. Why does randomising the order matter?

In [ ]:
# Solution — try it yourself first, then run this
strength = np.array([0.0, 0.5, 1.0, 1.5])
elo = np.full(4, 1000.0)
r7 = np.random.default_rng(7)
for _ in range(600):
    s1, s2 = r7.choice(4, 2, replace=False)                  # s1 is shown first (random order)
    p_first = 1 / (1 + np.exp(-(strength[s1] - strength[s2] + 0.8)))
    first_wins = r7.random() < p_first
    expected = 1 / (1 + 10 ** ((elo[s2] - elo[s1]) / 400))
    elo[s1] += 16 * (first_wins - expected); elo[s2] -= 16 * (first_wins - expected)
print("Elo:", elo.round(0), " ranking (best first):", np.argsort(-elo))
# With random order the position bias adds noise but no systematic advantage to any system.
# If system 3 were always shown first, the bias would inflate its rating. And remember that
# Elo is relative: it cannot tell you whether all four systems are bad (ev08).

## Say it in an interview

**30-second answer (`ev03`):** "I treat the judge as a measuring instrument that has to be calibrated. Known biases: position (fix by judging both orders and keeping agreeing verdicts), verbosity (tell it length isn't quality and test with padded copies), self-preference (use a different model family), leniency and scale drift (anchored rubric, categorical labels, or pairwise instead of 1–10), and sampling noise (temperature 0 or majority of 3). Then I label 50–100 cases by hand, report Cohen's kappa between judge and humans, and only use the judge if it's around 0.6 or higher. I pin the judge model version and prompt, and re-run that calibration set whenever either changes."

**Why trust an LLM judge at all (`ev02`)?** Discrimination is easier than generation. The judge gets advantages the generator lacked (rubric, reference, context). And, crucially, its agreement with humans is **measured**, not assumed. Frame it as a regression detector, not an oracle.

**Numbers to have ready:**
- κ = (p_o − p_e)/(1 − p_e). 90% raw agreement can be κ ≈ 0.44.
- The swap test costs 2×, n = 3 voting 3×, both 6×.
- The swap test and padding test need **no labels**. Kappa needs labels.
- In the MT-Bench paper (Zheng et al., 2023), even GPT-4 kept its verdict after a swap only about two-thirds of the time.

**Follow-ups:**
- *Pairwise or pointwise?* Pairwise for "is B better than A", pointwise with a rubric for monitoring levels. Pairwise gives a ranking, never an absolute level.
- *How do you catch judge drift?* A frozen calibration set with human labels, pinned model versions, and kappa tracked over time.
- *What if humans disagree?* Measure inter-annotator kappa first. If it is low, fix the labelling guideline before evaluating any model.

**Traps:**
- Starting with the judge before you have assertions and retrieval metrics (`ev01`).
- A 1–10 overall score with no rubric.
- Using the same model to generate and to judge.
- Averaging votes to "fix" a bias. Voting removes noise, not bias.

## Next

- [01 · LLM eval metrics](01_llm_eval_metrics_em_f1_bleu_rouge.ipynb): the cheap metrics the judge sits on top of.
- [03 · RAG evaluation: retrieval and faithfulness](03_rag_evaluation_retrieval_and_faithfulness.ipynb): judges for groundedness.
- [04 · Agent evaluation](04_agent_evaluation.ipynb): judging trajectories and final answers.
- [05 · A/B testing and statistics for evals](05_ab_testing_and_statistics_for_evals.ipynb): confidence intervals on judge scores.
- [Structured outputs with Pydantic](../08_llm_apps/03_structured_outputs_with_pydantic.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)